# iLQR for a Damped Cart-Pole

This notebook follows the structure of `ilqr_driving.ipynb`, but replaces the vehicle model with the damped cart-pole dynamics from the MATLAB code.

We stop after computing the optimized state trajectory `x_trj` and input trajectory `u_trj`. No visualization is included here.

In [1]:
import numpy as np
import sympy as sp
from dataclasses import dataclass

## Problem setup

The state and input are

$$
\mathbf{x}
=
\begin{bmatrix}
p & \theta & \dot p & \dot\theta
\end{bmatrix}^T,
\qquad
\mathbf{u}
=
\begin{bmatrix}
F
\end{bmatrix}.
$$

The goal is to swing the pole from

$$
\mathbf{x}_0
=
\begin{bmatrix}
0 & 0 & 0 & 0
\end{bmatrix}^T
$$

to the upright target

$$
\mathbf{x}_d
=
\begin{bmatrix}
0 & \pi & 0 & 0
\end{bmatrix}^T.
$$

As in the MATLAB code, the physical parameters are

$$
m_c=10,\quad
m_p=2,\quad
l=0.5,\quad
g=9.8,\quad
b=0.1,\quad
d=0.1.
$$

In [2]:
n_x = 4
n_u = 1

T = 2.5
dt = 0.01
N = int(T / dt)        # Number of state samples, matching the MATLAB convention.
N_u = N - 1            # Number of control intervals.

x0 = np.array([0.0, 0.0, 0.0, 0.0])
xd = np.array([0.0, np.pi, 0.0, 0.0])

# More balanced swing-up weights.
# The stage cost is target-centered, so these do not need to be extreme.
Q = np.diag([2.0, 2.0, 0.1, 0.1])
Qf = np.diag([200.0, 2000.0, 100.0, 300.0])
R = np.array([[0.003]])


## Cart-pole dynamics from the MATLAB code

Let

$$
D(\theta)=m_c+m_p\sin^2\theta.
$$

The continuous dynamics are

$$
\dot p = \dot p,
\qquad
\dot\theta = \dot\theta,
$$

and

$$
\ddot p
=
\frac{
u-b\dot p+\frac{d}{l}\dot\theta\cos\theta
+
m_p\sin\theta\left(l\dot\theta^2+g\cos\theta\right)
}{
D(\theta)
},
$$

$$
\ddot\theta
=
\frac{
-u\cos\theta
+b\dot p\cos\theta
-\frac{d(m_c+m_p)}{m_p l}\dot\theta
-m_p l\dot\theta^2\cos\theta\sin\theta
-\frac{(m_c+m_p)g}{m_p l}\sin\theta
}{
lD(\theta)
}.
$$

The discrete dynamics use the same simple Euler step as the driving notebook:

$$
\mathbf{x}[n+1]
=
\mathbf{f}_d(\mathbf{x}[n],\mathbf{u}[n])
=
\mathbf{x}[n]+h\mathbf{f}_c(\mathbf{x}[n],\mathbf{u}[n]).
$$

As in `ilqr_driving.ipynb`, we do not hand-code the Jacobians. We write the dynamics once, build symbolic derivatives once, and then evaluate those derivatives numerically inside the iLQR backward pass.


In [3]:
@dataclass
class CartPoleParams:
    mc: float = 10.0
    mp: float = 2.0
    l: float = 0.5
    g: float = 9.8
    b: float = 0.1
    d: float = 0.1


class CartPoleModel:
    """Damped cart-pole model using the MATLAB dynamics."""

    def __init__(self, params: CartPoleParams, dt: float):
        self.params = params
        self.dt = dt

    @staticmethod
    def _is_symbolic(x, u):
        return np.asarray(x).dtype == object or np.asarray(u).dtype == object

    def continuous_dynamics(self, x, u):
        symbolic = self._is_symbolic(x, u)
        x = np.asarray(x, dtype=object if symbolic else float)
        u = np.asarray(u, dtype=object if symbolic else float).reshape(-1)[0]
        m = sp if symbolic else np

        mc, mp, l, g, b, d = self.params.mc, self.params.mp, self.params.l, self.params.g, self.params.b, self.params.d
        _, theta, p_dot, theta_dot = x
        s, c = m.sin(theta), m.cos(theta)
        D = mc + mp * s**2

        x_dot = np.array([
            p_dot,
            theta_dot,
            (u - b * p_dot + d * theta_dot * c / l + mp * s * (l * theta_dot**2 + g * c)) / D,
            (-u * c + b * p_dot * c - d * (mc + mp) * theta_dot / (mp * l) - mp * l * theta_dot**2 * c * s - (mc + mp) * g * s / (mp * l)) / (l * D),
        ], dtype=object if symbolic else float)

        return x_dot

    def discrete_dynamics(self, x, u):
        symbolic = self._is_symbolic(x, u)
        x = np.asarray(x, dtype=object if symbolic else float)
        return x + self.dt * self.continuous_dynamics(x, u)


model = CartPoleModel(CartPoleParams(), dt)

## Cost functions

For swing-up, the running cost should not keep pulling the pole back toward the downward state.
Therefore this notebook uses a target-centered quadratic stage cost,

$$
\ell(\mathbf{x},\mathbf{u})
=
\frac{1}{2}(\mathbf{x}-\mathbf{x}_d)^TQ(\mathbf{x}-\mathbf{x}_d)
+
\frac{1}{2}\mathbf{u}^TR\mathbf{u},
$$

and the terminal cost is

$$
\ell_f(\mathbf{x})
=
\frac{1}{2}(\mathbf{x}-\mathbf{x}_d)^TQ_f(\mathbf{x}-\mathbf{x}_d).
$$

Therefore

$$
\ell_{\mathbf{x}}=Q(\mathbf{x}-\mathbf{x}_d),
\qquad
\ell_{\mathbf{u}}=R\mathbf{u},
\qquad
\ell_{\mathbf{xx}}=Q,
\qquad
\ell_{\mathbf{ux}}=0,
\qquad
\ell_{\mathbf{uu}}=R,
$$

and

$$
\ell_{f,\mathbf{x}}=Q_f(\mathbf{x}-\mathbf{x}_d),
\qquad
\ell_{f,\mathbf{xx}}=Q_f.
$$

This is the main difference from the quick MATLAB script. If the stage cost is instead
$\frac{1}{2}\mathbf{x}^TQ\mathbf{x}$, then normal-sized $Q,Q_f,R$ tend to fight the swing-up, because being near
$\theta=\pi$ is penalized throughout the trajectory.


In [4]:
def cost_stage(x, u):
    x = np.asarray(x)
    u = np.asarray(u).reshape(n_u)
    e = x - xd
    return 0.5 * e @ Q @ e + 0.5 * u @ R @ u


def cost_final(x):
    x = np.asarray(x)
    e = x - xd
    return 0.5 * e @ Qf @ e


class CartPoleCost:
    """Target-centered quadratic running cost and terminal cost."""

    def stage(self, x, u):
        return float(cost_stage(np.asarray(x, dtype=float), np.asarray(u, dtype=float)))

    def final(self, x):
        return float(cost_final(np.asarray(x, dtype=float)))


cost = CartPoleCost()


def rollout(model, x0, u_trj):
    x_trj = np.zeros((u_trj.shape[0] + 1, n_x))
    x_trj[0, :] = x0

    for n in range(u_trj.shape[0]):
        x_trj[n + 1, :] = model.discrete_dynamics(x_trj[n, :], u_trj[n, :])

    return x_trj


def cost_trj(cost, x_trj, u_trj):
    total = 0.0

    for n in range(u_trj.shape[0]):
        total += cost.stage(x_trj[n, :], u_trj[n, :])

    total += cost.final(x_trj[-1, :])
    return float(total)


## Symbolic derivatives

This is the cart-pole version of the `class derivatives` pattern in `ilqr_driving.ipynb`: create symbolic state and input variables, build the cost gradients/Hessians and dynamics Jacobians once, then evaluate them numerically during the backward pass.

The solver uses derivatives of the discrete dynamics directly:

$$
\mathbf{f}_{\mathbf{x},n}
=
\frac{\partial \mathbf{f}_d}{\partial \mathbf{x}},
\qquad
\mathbf{f}_{\mathbf{u},n}
=
\frac{\partial \mathbf{f}_d}{\partial \mathbf{u}}.
$$


In [5]:
class Derivatives:
    def __init__(self, discrete_dynamics, cost_stage, cost_final, n_x, n_u):
        self.x_sym = np.array(sp.symbols(f"x_0:{n_x}"), dtype=object)
        self.u_sym = np.array(sp.symbols(f"u_0:{n_u}"), dtype=object)

        x_vec = sp.Matrix(self.x_sym)
        u_vec = sp.Matrix(self.u_sym)
        args = tuple(self.x_sym) + tuple(self.u_sym)

        l = sp.sympify(cost_stage(self.x_sym, self.u_sym))
        l_x = sp.Matrix([sp.diff(l, x_i) for x_i in self.x_sym])
        l_u = sp.Matrix([sp.diff(l, u_i) for u_i in self.u_sym])
        l_xx = l_x.jacobian(x_vec)
        l_ux = l_u.jacobian(x_vec)
        l_uu = l_u.jacobian(u_vec)

        l_final = sp.sympify(cost_final(self.x_sym))
        l_final_x = sp.Matrix([sp.diff(l_final, x_i) for x_i in self.x_sym])
        l_final_xx = l_final_x.jacobian(x_vec)

        f = sp.Matrix(discrete_dynamics(self.x_sym, self.u_sym))
        f_x = f.jacobian(x_vec)
        f_u = f.jacobian(u_vec)

        self._stage_fn = sp.lambdify(args, [l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u], modules="numpy")
        self._final_fn = sp.lambdify(tuple(self.x_sym), [l_final_x, l_final_xx], modules="numpy")

    def stage(self, x, u):
        values = self._stage_fn(*np.asarray(x, dtype=float).reshape(n_x), *np.asarray(u, dtype=float).reshape(n_u))
        l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u = [np.asarray(v, dtype=float) for v in values]
        return l_x.ravel(), l_u.ravel(), l_xx, l_ux, l_uu, f_x, f_u

    def final(self, x):
        values = self._final_fn(*np.asarray(x, dtype=float).reshape(n_x))
        l_final_x, l_final_xx = [np.asarray(v, dtype=float) for v in values]
        return l_final_x.ravel(), l_final_xx


derivs = Derivatives(model.discrete_dynamics, cost_stage, cost_final, n_x, n_u)


## Bellman recursion and Q-function expansion

The finite-horizon optimal control problem is

$$
\begin{aligned}
\min_{\mathbf{u}[\cdot]}
\quad&
\ell_f(\mathbf{x}[N])
+
\sum_{n=0}^{N-1}
\ell(\mathbf{x}[n],\mathbf{u}[n])
\\
\text{subject to}\quad&
\mathbf{x}[n+1]=\mathbf{f}(\mathbf{x}[n],\mathbf{u}[n]),
\\
&\mathbf{x}[0]=\mathbf{x}_0.
\end{aligned}
$$

The Bellman recursion is

$$
V(\mathbf{x}[n])
=
\min_{\mathbf{u}[n]}
\left[
\ell(\mathbf{x}[n],\mathbf{u}[n])
+
V(\mathbf{x}[n+1])
\right].
$$

Define the local Q-function

$$
Q(\mathbf{x}[n],\mathbf{u}[n])
=
\ell(\mathbf{x}[n],\mathbf{u}[n])
+
V(\mathbf{f}(\mathbf{x}[n],\mathbf{u}[n])).
$$

Around the nominal trajectory, write

$$
\delta\mathbf{x}
=
\mathbf{x}[n]-\bar{\mathbf{x}}[n],
\qquad
\delta\mathbf{u}
=
\mathbf{u}[n]-\bar{\mathbf{u}}[n].
$$

Using the linearized dynamics

$$
\delta\mathbf{x}[n+1]
\approx
\mathbf{f}_{\mathbf{x},n}\delta\mathbf{x}
+
\mathbf{f}_{\mathbf{u},n}\delta\mathbf{u},
$$

the Q-terms are

$$
\begin{aligned}
Q_{\mathbf{x}}
&=
\ell_{\mathbf{x}}
+
\mathbf{f}_{\mathbf{x}}^T V_{\mathbf{x}},
\\
Q_{\mathbf{u}}
&=
\ell_{\mathbf{u}}
+
\mathbf{f}_{\mathbf{u}}^T V_{\mathbf{x}},
\\
Q_{\mathbf{xx}}
&=
\ell_{\mathbf{xx}}
+
\mathbf{f}_{\mathbf{x}}^T V_{\mathbf{xx}}\mathbf{f}_{\mathbf{x}},
\\
Q_{\mathbf{ux}}
&=
\ell_{\mathbf{ux}}
+
\mathbf{f}_{\mathbf{u}}^T V_{\mathbf{xx}}\mathbf{f}_{\mathbf{x}},
\\
Q_{\mathbf{uu}}
&=
\ell_{\mathbf{uu}}
+
\mathbf{f}_{\mathbf{u}}^T V_{\mathbf{xx}}\mathbf{f}_{\mathbf{u}}.
\end{aligned}
$$

In [6]:
def Q_terms(l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u, V_x, V_xx):
    Q_x = l_x + f_x.T @ V_x
    Q_u = l_u + f_u.T @ V_x
    Q_xx = l_xx + f_x.T @ V_xx @ f_x
    Q_ux = l_ux + f_u.T @ V_xx @ f_x
    Q_uu = l_uu + f_u.T @ V_xx @ f_u

    return Q_x, Q_u, Q_xx, Q_ux, Q_uu

## Gains, value update, and expected cost reduction

The local quadratic Q-function is minimized over $\delta\mathbf{u}$ while holding $\delta\mathbf{x}$ fixed:

$$
\frac{\partial Q}{\partial \delta\mathbf{u}}
=
Q_{\mathbf{u}}
+
Q_{\mathbf{ux}}\delta\mathbf{x}
+
Q_{\mathbf{uu}}\delta\mathbf{u}
=
0.
$$

Hence

$$
\delta\mathbf{u}^*
=
k+K\delta\mathbf{x},
$$

where

$$
k
=
-Q_{\mathbf{uu}}^{-1}Q_{\mathbf{u}},
\qquad
K
=
-Q_{\mathbf{uu}}^{-1}Q_{\mathbf{ux}}.
$$

Substituting $\delta\mathbf{u}^*=k+K\delta\mathbf{x}$ back into the Q-function gives the backward update

$$
V_{\mathbf{x}}
=
Q_{\mathbf{x}}
+
K^TQ_{\mathbf{u}}
+
Q_{\mathbf{ux}}^Tk
+
K^TQ_{\mathbf{uu}}k,
$$

$$
V_{\mathbf{xx}}
=
Q_{\mathbf{xx}}
+
K^TQ_{\mathbf{ux}}
+
Q_{\mathbf{ux}}^TK
+
K^TQ_{\mathbf{uu}}K.
$$

The expected local cost reduction is

$$
\Delta J_{\mathrm{expected}}
=
-Q_{\mathbf{u}}^Tk
-
\frac{1}{2}k^TQ_{\mathbf{uu}}k.
$$

In [7]:
def gains(Q_uu, Q_u, Q_ux):
    k = -np.linalg.solve(Q_uu, Q_u)
    K = -np.linalg.solve(Q_uu, Q_ux)
    return k, K


def V_terms(Q_x, Q_u, Q_xx, Q_ux, Q_uu, K, k):
    V_x = Q_x + K.T @ Q_u + Q_ux.T @ k + K.T @ Q_uu @ k
    V_xx = Q_xx + K.T @ Q_ux + Q_ux.T @ K + K.T @ Q_uu @ K

    # Symmetrize for numerical hygiene.
    V_xx = 0.5 * (V_xx + V_xx.T)

    return V_x, V_xx


def expected_cost_reduction(Q_u, Q_uu, k):
    return float(-Q_u.T @ k - 0.5 * k.T @ Q_uu @ k)

## Forward pass, backward pass, and line search

The backward pass computes local gains around the nominal trajectory:

$$
\delta\mathbf{u}[n]
=
k[n]+K[n]\delta\mathbf{x}[n].
$$

The forward pass rolls out the nonlinear dynamics with

$$
\mathbf{u}'[n]
=
\bar{\mathbf{u}}[n]
+
\alpha k[n]
+
K[n]\left(\mathbf{x}'[n]-\bar{\mathbf{x}}[n]\right).
$$

Here $\alpha$ is the explicit line-search step size. The implementation keeps the main numerical trick from
`ilqr_driving.ipynb`: regularize the control Hessian before solving for $k$ and $K$,

$$
Q_{\mathbf{uu},n}^{\mathrm{reg}}
=
Q_{\mathbf{uu},n}
+
\rho I.
$$

Larger $\rho$ makes the inverse better conditioned and makes the feedforward step smaller. Thus the solver uses both safeguards:

1. increase $\rho$ when the backward pass or rollout is too aggressive;
2. try $\alpha=1,0.5,0.25,\ldots$ until the rollout decreases the total cost.

For this cart-pole problem, this is enough to use $dt=0.01$ and still converge cleanly with moderate cost weights.


In [8]:
class ILQRSolver:
    def __init__(
        self,
        model,
        cost,
        derivs,
        max_iter=300,
        regu_init=1.0,
        min_regu=1e-9,
        max_regu=1e10,
        alphas=(1.0, 0.5, 0.25, 0.1, 0.05, 0.025, 0.01, 0.005, 0.001),
        tol=1e-7,
        u_clip=None,
    ):
        self.model = model
        self.cost = cost
        self.derivs = derivs
        self.max_iter = max_iter
        self.regu_init = regu_init
        self.min_regu = min_regu
        self.max_regu = max_regu
        self.alphas = alphas
        self.tol = tol
        self.u_clip = u_clip

    def backward_pass(self, x_trj, u_trj, regu):
        k_trj = np.zeros((u_trj.shape[0], n_u))
        K_trj = np.zeros((u_trj.shape[0], n_u, n_x))
        expected_reduction = 0.0

        V_x, V_xx = self.derivs.final(x_trj[-1, :])

        for n in range(u_trj.shape[0] - 1, -1, -1):
            l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u = self.derivs.stage(
                x_trj[n, :], u_trj[n, :]
            )

            Q_x, Q_u, Q_xx, Q_ux, Q_uu = Q_terms(
                l_x, l_u, l_xx, l_ux, l_uu, f_x, f_u, V_x, V_xx
            )

            Q_uu_reg = Q_uu + regu * np.eye(n_u)
            k, K = gains(Q_uu_reg, Q_u, Q_ux)

            k_trj[n, :] = k
            K_trj[n, :, :] = K

            # Match the driving notebook: gains use regularized Q_uu,
            # while the value update keeps the original quadratic model.
            V_x, V_xx = V_terms(Q_x, Q_u, Q_xx, Q_ux, Q_uu, K, k)
            expected_reduction += expected_cost_reduction(Q_u, Q_uu, k)

            if not (np.all(np.isfinite(V_x)) and np.all(np.isfinite(V_xx))):
                raise FloatingPointError("Backward pass produced non-finite value derivatives.")

        return k_trj, K_trj, expected_reduction

    def forward_pass(self, x_trj, u_trj, k_trj, K_trj, alpha):
        x_new = np.zeros_like(x_trj)
        u_new = np.zeros_like(u_trj)
        x_new[0, :] = x_trj[0, :]

        for n in range(u_trj.shape[0]):
            dx = x_new[n, :] - x_trj[n, :]
            u_new[n, :] = u_trj[n, :] + alpha * k_trj[n, :] + K_trj[n, :, :] @ dx

            if self.u_clip is not None:
                u_new[n, :] = np.clip(u_new[n, :], -self.u_clip, self.u_clip)

            x_new[n + 1, :] = self.model.discrete_dynamics(x_new[n, :], u_new[n, :])

            if not (np.all(np.isfinite(x_new[n + 1, :])) and np.all(np.isfinite(u_new[n, :]))):
                raise FloatingPointError("Forward rollout produced non-finite values.")

        return x_new, u_new

    def solve(self, x0, N_u, u_init=None):
        if u_init is None:
            u_trj = np.zeros((N_u, n_u))
        else:
            u_trj = np.asarray(u_init, dtype=float).reshape(N_u, n_u)

        x_trj = rollout(self.model, x0, u_trj)
        J = cost_trj(self.cost, x_trj, u_trj)
        regu = self.regu_init

        trace = {
            "cost": [J],
            "regu": [regu],
            "alpha": [],
            "expected_reduction": [],
            "actual_reduction": [],
            "accepted": [],
        }

        for it in range(self.max_iter):
            try:
                k_trj, K_trj, expected_reduction = self.backward_pass(x_trj, u_trj, regu)
            except (np.linalg.LinAlgError, FloatingPointError):
                regu = min(self.max_regu, 10.0 * regu)
                trace["accepted"].append(False)
                continue

            accepted = False

            for alpha in self.alphas:
                try:
                    x_candidate, u_candidate = self.forward_pass(
                        x_trj, u_trj, k_trj, K_trj, alpha
                    )
                    J_candidate = cost_trj(self.cost, x_candidate, u_candidate)
                except FloatingPointError:
                    continue

                actual_reduction = J - J_candidate

                if np.isfinite(J_candidate) and actual_reduction > 0.0:
                    x_trj = x_candidate
                    u_trj = u_candidate
                    J = J_candidate
                    regu = max(self.min_regu, 0.5 * regu)

                    trace["cost"].append(J)
                    trace["regu"].append(regu)
                    trace["alpha"].append(alpha)
                    trace["expected_reduction"].append(expected_reduction)
                    trace["actual_reduction"].append(actual_reduction)
                    trace["accepted"].append(True)

                    accepted = True
                    break

            if not accepted:
                regu = min(self.max_regu, 10.0 * regu)
                trace["cost"].append(J)
                trace["regu"].append(regu)
                trace["alpha"].append(0.0)
                trace["expected_reduction"].append(expected_reduction)
                trace["actual_reduction"].append(0.0)
                trace["accepted"].append(False)

            if accepted and abs(trace["actual_reduction"][-1]) < self.tol:
                break

        return x_trj, u_trj, trace

## Run iLQR

The final result is stored in:

- `x_trj`: optimized state trajectory, shape `(N, 4)`;
- `u_trj`: optimized input trajectory, shape `(N-1, 1)`;
- `trace`: cost, regularization, accepted step sizes, and reduction history.

This is where the notebook stops.

In [9]:
solver = ILQRSolver(
    model=model,
    cost=cost,
    derivs=derivs,
    max_iter=250,
    regu_init=1.0,
    tol=1e-6,
)

x_trj, u_trj, trace = solver.solve(x0=x0, N_u=N_u)

print("Number of accepted steps:", sum(trace["accepted"]))
print("Number of total iterations:", len(trace["accepted"]))
print("Initial cost:", trace["cost"][0])
print("Final cost:", trace["cost"][-1])
print("Final state x[N-1]:", x_trj[-1])
print("Target state xd:", xd)
print("Final error x[N-1] - xd:", x_trj[-1] - xd)
print("First five controls:")
print(u_trj[:5].ravel())
print("Last five controls:")
print(u_trj[-5:].ravel())
print("Max |u|:", np.max(np.abs(u_trj)))


Number of accepted steps: 196
Number of total iterations: 196
Initial cost: 12327.135896960612
Final cost: 2163.8337455585483
Final state x[N-1]: [-1.03476792e-01  3.14177424e+00  6.75200288e-02 -1.84361770e-03]
Target state xd: [0.         3.14159265 0.         0.        ]
Final error x[N-1] - xd: [-0.10347679  0.00018159  0.06752003 -0.00184362]
First five controls:
[81.38716335 81.76352558 81.95353959 81.95583567 81.76858216]
Last five controls:
[-1.62241731 -1.68788866 -1.75294389 -1.81761743 -1.88194408]
Max |u|: 149.826583886831


## Alternative terminal cost: local infinite-horizon LQR value

The previous run used a hand-chosen diagonal terminal matrix `Qf`.  A more principled version is to compute the terminal value from the local infinite-horizon LQR problem around the upright equilibrium.

Linearize the discrete dynamics at

$$
\mathbf{x}^\star = \begin{bmatrix}0 & \pi & 0 & 0\end{bmatrix}^T,
\qquad
\mathbf{u}^\star = 0,
$$

then solve the discrete algebraic Riccati equation

$$
S_\infty
=
Q_{\mathrm{lqr}}
+
A^T S_\infty A
-
A^T S_\infty B
\left(R_{\mathrm{lqr}}+B^T S_\infty B\right)^{-1}
B^T S_\infty A.
$$

With the notebook's convention

$$
\ell_f(\mathbf{x}_N)
=
\frac{1}{2}
(\mathbf{x}_N-\mathbf{x}^\star)^T
S_\infty
(\mathbf{x}_N-\mathbf{x}^\star),
$$

the terminal derivatives are

$$
V_{\mathbf{x},N}=S_\infty(\mathbf{x}_N-\mathbf{x}^\star),
\qquad
V_{\mathbf{xx},N}=S_\infty.
$$

The interpretation is clean: iLQR handles the nonlinear swing-up, and the terminal value approximates the remaining infinite-horizon stabilization cost near the upright fixed point.

In [10]:
from scipy.linalg import solve_discrete_are

# Upright equilibrium for the damped cart-pole.
x_star = xd.copy()
u_star = np.zeros(n_u)

# Linearize the discrete-time dynamics x[n+1] = f_d(x[n], u[n]) at the upright equilibrium.
# The current Derivatives object already contains the symbolic dynamics Jacobians.
_, _, _, _, _, A_eq, B_eq = derivs.stage(x_star, u_star)

# These are local stabilization weights, not global swing-up tuning weights.
Q_lqr = np.diag([2.0, 20.0, 1.0, 5.0])
R_lqr = np.array([[0.003]])

S_inf = solve_discrete_are(A_eq, B_eq, Q_lqr, R_lqr)
S_inf = 0.5 * (S_inf + S_inf.T)  # remove tiny numerical asymmetry

print("A_eq:")
print(A_eq)
print("B_eq:")
print(B_eq)
print("diag(S_inf):", np.diag(S_inf))


A_eq:
[[ 1.000e+00  0.000e+00  1.000e-02  0.000e+00]
 [ 0.000e+00  1.000e+00  0.000e+00  1.000e-02]
 [ 0.000e+00  1.960e-02  9.999e-01 -2.000e-04]
 [ 0.000e+00  2.352e-01 -2.000e-04  9.976e-01]]
B_eq:
[[0.   ]
 [0.   ]
 [0.001]
 [0.002]]
diag(S_inf): [ 349.48843161 7439.44565994  356.78587699  282.29774601]


Now rerun iLQR with exactly the same dynamics, stage cost, backward pass, forward pass, regularization, and line search.  The only change is that the terminal cost uses `S_inf` instead of a manually chosen diagonal `Qf`.

In [11]:
def cost_final_lqr_terminal(x):
    x = np.asarray(x)
    e = x - x_star
    return 0.5 * e @ S_inf @ e


class CartPoleCostWithLQRTerminal:
    """Same target-centered stage cost, but terminal value from local infinite-horizon LQR."""

    def stage(self, x, u):
        return float(cost_stage(np.asarray(x, dtype=float), np.asarray(u, dtype=float)))

    def final(self, x):
        return float(cost_final_lqr_terminal(np.asarray(x, dtype=float)))


cost_lqr_terminal = CartPoleCostWithLQRTerminal()
derivs_lqr_terminal = Derivatives(
    model.discrete_dynamics,
    cost_stage,
    cost_final_lqr_terminal,
    n_x,
    n_u,
)

solver_lqr_terminal = ILQRSolver(
    model=model,
    cost=cost_lqr_terminal,
    derivs=derivs_lqr_terminal,
    max_iter=250,
    regu_init=1.0,
    tol=1e-6,
)

x_trj_lqr_terminal, u_trj_lqr_terminal, trace_lqr_terminal = solver_lqr_terminal.solve(
    x0=x0,
    N_u=N_u,
)

print("Number of accepted steps:", sum(trace_lqr_terminal["accepted"]))
print("Number of total iterations:", len(trace_lqr_terminal["accepted"]))
print("Initial cost:", trace_lqr_terminal["cost"][0])
print("Final cost:", trace_lqr_terminal["cost"][-1])
print("Final state x[N-1]:", x_trj_lqr_terminal[-1])
print("Target state xd:", xd)
print("Final error x[N-1] - xd:", x_trj_lqr_terminal[-1] - xd)
print("First five controls:")
print(u_trj_lqr_terminal[:5].ravel())
print("Last five controls:")
print(u_trj_lqr_terminal[-5:].ravel())
print("Max |u|:", np.max(np.abs(u_trj_lqr_terminal)))


Number of accepted steps: 210
Number of total iterations: 210
Initial cost: 39169.724309366284
Final cost: 2163.29159840797
Final state x[N-1]: [-0.10757839  3.14196872  0.10491469  0.0284578 ]
Target state xd: [0.         3.14159265 0.         0.        ]
Final error x[N-1] - xd: [-0.10757839  0.00037606  0.10491469  0.0284578 ]
First five controls:
[81.38001721 81.75070211 81.93455347 81.93021609 81.7358725 ]
Last five controls:
[-0.79787103 -0.86261173 -0.92755452 -0.99277138 -1.05833663]
Max |u|: 149.9468657786298


## Visualize the optimized cart-pole swing-up trajectories

The reference cart-pole balancing notebook builds a visualizer, runs the system, and displays the result with `display(HTML(...to_jshtml()))`.  Here we use the same notebook-style output, but we do **not** switch to Drake's undamped cart-pole model.  The two trajectories below are the trajectories already optimized using the MATLAB damped dynamics and parameters:

- fixed diagonal terminal matrix `Qf`, stored in `x_trj`, `u_trj`;
- local infinite-horizon LQR terminal value `S_inf`, stored in `x_trj_lqr_terminal`, `u_trj_lqr_terminal`.


In [ ]:
from IPython.display import HTML, display
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle, Circle
from matplotlib.lines import Line2D


def animate_cartpole_trajectory(x_trj, title, params, dt, stride=5):
    """Animate a precomputed cart-pole trajectory.

    This mirrors the reference notebook style: construct an animation object,
    convert it to JavaScript HTML, and display it directly below the cell.
    The geometry uses the same coordinate convention as the MATLAB draw_cartpole:

        pole tip = (q + l sin(theta), -l cos(theta)).

    Therefore theta = 0 is downward and theta = pi is upright.
    """
    x_trj = np.asarray(x_trj, dtype=float)
    l = params.l

    frame_ids = np.arange(0, x_trj.shape[0], stride)
    if frame_ids[-1] != x_trj.shape[0] - 1:
        frame_ids = np.r_[frame_ids, x_trj.shape[0] - 1]

    q_min = np.min(x_trj[:, 0])
    q_max = np.max(x_trj[:, 0])
    x_margin = max(0.7, 0.25 * (q_max - q_min + 1e-9))

    cart_w = 0.30
    cart_h = 0.15
    wheel_r = 0.05

    fig, ax = plt.subplots(figsize=(6.2, 2.8), dpi=90)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(q_min - x_margin, q_max + x_margin)
    ax.set_ylim(-1.35 * l, 1.45 * l)
    ax.set_xlabel("cart position q")
    ax.set_yticks([])
    ax.grid(True, axis="x", alpha=0.25)
    ax.set_title(title)

    rail = Line2D([q_min - x_margin, q_max + x_margin], [-cart_h / 2 - wheel_r, -cart_h / 2 - wheel_r], linewidth=1.0)
    ax.add_line(rail)

    cart = Rectangle((-cart_w / 2, -cart_h / 2), cart_w, cart_h)
    left_wheel = Circle((-cart_w / 3, -cart_h / 2 - wheel_r), wheel_r)
    right_wheel = Circle((cart_w / 3, -cart_h / 2 - wheel_r), wheel_r)
    pole = Line2D([], [], linewidth=4)
    bob = Circle((0.0, 0.0), 0.065)
    time_text = ax.text(0.02, 0.90, "", transform=ax.transAxes)

    for artist in (cart, left_wheel, right_wheel, bob):
        ax.add_patch(artist)
    ax.add_line(pole)

    def update(frame_id):
        q, theta, q_dot, theta_dot = x_trj[frame_id]
        tip_x = q + l * np.sin(theta)
        tip_y = -l * np.cos(theta)

        cart.set_xy((q - cart_w / 2, -cart_h / 2))
        left_wheel.center = (q - cart_w / 3, -cart_h / 2 - wheel_r)
        right_wheel.center = (q + cart_w / 3, -cart_h / 2 - wheel_r)
        pole.set_data([q, tip_x], [0.0, tip_y])
        bob.center = (tip_x, tip_y)
        time_text.set_text(f"t = {frame_id * dt:.2f} s")

        return cart, left_wheel, right_wheel, pole, bob, time_text

    ani = FuncAnimation(
        fig,
        update,
        frames=frame_ids,
        interval=1000 * dt * stride,
        blit=True,
    )

    display(HTML(ani.to_jshtml()))
    plt.close(fig)
    return ani


In [ ]:
# Visualize the trajectory from the fixed diagonal terminal matrix Qf.
ani_fixed_S = animate_cartpole_trajectory(
    x_trj=x_trj,
    title="iLQR swing-up with fixed diagonal terminal S = Qf",
    params=model.params,
    dt=dt,
    stride=5,
)

# Visualize the trajectory from the local infinite-horizon LQR terminal value S_inf.
ani_S_inf = animate_cartpole_trajectory(
    x_trj=x_trj_lqr_terminal,
    title="iLQR swing-up with local infinite-horizon LQR terminal S_inf",
    params=model.params,
    dt=dt,
    stride=5,
)
